# 02b. 일관성 점검 (사실 확인)

**판정하지 않는다.** CoT 안의 서로 다른 부분이 같은 사실을 같게 적고 있는지, 데이터셋에 중복·편중이 있는지 **건수와 샘플**만 확인한다.
품질 기준 판정은 03 단계(판정 함수)에서 한다.

| # | 항목 | 비교 대상 |
|---|---|---|
| 1 | 환경 | git이 원본을 추적하지 않는지 |
| 2 | 상품 등급 | cot3에 적힌 상품별 등급 vs 상품 JSON `risk_grade` |
| 3 | 성향별 허용 등급 | 투자성향(cot1/cot2) × cot2 허용 등급 집합 |
| 4 | 추천 상품명 | answer 본문 vs `product_names` |
| 5 | 메타 vs 본문 | 메타 age·gender vs question 본문 |
| 6 | 고령자 고위험 | 70대 이상 × 1~2등급 추천 |
| 7 | 중복 | question 완전·유사 중복 |
| 8 | 추천 편중 | 상품별 추천 횟수 |
| 9 | 문장 완결 | answer 끝 문장 부호 |

추출 함수는 `src/extract.py`, 상품명 매칭은 D10 확정 규칙(정확 → 정규화 → 보험 모상품명)을 따른다.

In [ ]:
# 라이브러리와 프로젝트 함수 불러오기
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.load_cot import load_cot, load_products
from src.extract import (build_product_index, match_product, product_grade, parse_allowed_grades,
                         grade_after_name, propensity_in, extract_age_decade, extract_gender,
                         meta_age_match, quoted_strings, ends_with_punct, norm_full,
                         near_duplicate_groups, PROPENSITY)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

In [ ]:
# 데이터 읽기: CoT 전체(Train+Valid), 상품 JSON(Training 폴더, D1)
cot = load_cot()
prod = load_products()
idx = build_product_index(prod)
print("CoT:", cot.shape, "/ 상품:", prod.shape)

In [ ]:
# 추천 상품명을 한 행에 하나씩 펼치고 D10 규칙으로 상품 JSON에 매칭
rec = (cot[["split", "query_group", "category", "cot_id", "age", "product_names"]]
       .explode("product_names").dropna(subset=["product_names"])
       .rename(columns={"product_names": "rec_name"}).reset_index(drop=True))
m = rec["rec_name"].map(lambda n: match_product(n, idx))
rec["match_stage"] = m.str[0]
rec["keys"] = m.str[1]
rec["json_grade"] = rec["keys"].map(lambda k: product_grade(k, idx))
print("추천 상품명 수:", len(rec))
rec["match_stage"].value_counts()

## 1. 환경: git이 원본을 추적하지 않는지

In [ ]:
# data/raw/ 아래 파일이 .gitignore로 제외되는지 확인 (제외되면 규칙과 파일명이 출력됨)
r = subprocess.run(["git", "-C", str(ROOT), "check-ignore", "-v", "data/raw/Training/02.라벨링데이터/TL_2.소비자.zip"],
                   capture_output=True, text=True, encoding="utf-8")
print("check-ignore:", r.stdout.strip() or "(제외되지 않음)")

# git이 추적 중이거나 추가 대상으로 보는 파일 중 data/ 아래 파일 수
r = subprocess.run(["git", "-C", str(ROOT), "-c", "core.quotepath=false", "status", "--porcelain", "-uall"],
                   capture_output=True, text=True, encoding="utf-8")
tracked_data = [l for l in r.stdout.splitlines() if " data/" in l]
print("git status에 나온 data/ 파일 수:", len(tracked_data))

# 노트북 출력 제거 필터(nbstripout) 설정 확인
print(open(ROOT / ".gitattributes", encoding="utf-8").read())

## 2. cot3에 적힌 상품별 등급 vs 상품 JSON `risk_grade` (증권)

- 샘플 확인 결과 cot2에는 상품명이 나오지 않아(0%), 상품별 등급은 **cot3**에서 뽑는다.
- 범위 표현("N등급 이상/이하", "N~M등급", "N·M등급")은 제외한다.
- 추출 실패는 "상품명 뒤에 등급 표현 자체가 없음"과 "있지만 추출 실패(범위 표현만 있음)"로 나눈다.

In [ ]:
# 증권 추천 상품별로 cot3에서 등급을 뽑는다
sec = rec[rec["category"] == "증권"].merge(cot[["cot_id", "split", "cot3", "product_names"]], on=["cot_id", "split"])
res = sec.apply(lambda r: grade_after_name(r["cot3"], r["rec_name"], r["product_names"]), axis=1)
sec["cot3_status"] = res.str[0]
sec["cot3_grade"] = res.str[1]
sec["cot3_status"].value_counts().to_frame("건수")

In [ ]:
# 추출 성공 건에서 JSON 등급과 비교
# 비교 불가: 상품 JSON 매칭 실패 또는 이름 중복으로 등급이 여러 개인 경우
ok = sec[sec["cot3_status"] == "추출"].copy()
ok["비교"] = np.select(
    [ok["json_grade"].isna(), ok["cot3_grade"] == ok["json_grade"]],
    ["비교 불가(JSON 등급 없음)", "일치"], default="불일치")
print(ok["비교"].value_counts())
print()
# 불일치 건: cot3 등급 × JSON 등급 교차표
mis = ok[ok["비교"] == "불일치"]
pd.crosstab(mis["cot3_grade"], mis["json_grade"], margins=True)

In [ ]:
# 불일치 샘플 10건: 상품명 뒤 구간의 앞 120자를 함께 본다
def after_name(text, name, n=120):
    i = str(text).find(name)
    return str(text)[i + len(name): i + len(name) + n] if i >= 0 else ""

s = mis.sample(min(10, len(mis)), random_state=0)
for _, r in s.iterrows():
    print(f"[{r['split']} cot_id={r['cot_id']}] {r['rec_name']}")
    print(f"   cot3 등급={r['cot3_grade']:.0f} / JSON 등급={r['json_grade']:.0f}")
    print("   상품명 뒤:", after_name(r["cot3"], r["rec_name"]).replace("\n", " "))

In [ ]:
# 추출 실패 샘플: '등급 표현 없음' 5건, '있지만 추출 실패' 5건 (상품명 뒤 150자)
for status in ["등급 표현 없음", "있지만 추출 실패"]:
    sub = sec[sec["cot3_status"] == status]
    print("=" * 100)
    print(status, f"({len(sub)}건)")
    for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
        print(f"[cot_id={r['cot_id']}] {r['rec_name']}")
        print("   상품명 뒤:", after_name(r["cot3"], r["rec_name"], 150).replace("\n", " "))

## 3. 투자성향별 cot2 허용 등급 표기

- 성향은 cot1 기준과 cot2 기준을 나눠 뽑는다 (증권_03은 cot2에 성향이 나온다).
- 성향 명칭이 2개 이상이면 '복수', 없으면 '없음'으로 둔다.

In [ ]:
# 증권 CoT만 대상 (보험 cot2에는 등급 표현이 거의 없음: 02 노트북 6-3)
sc = cot[cot["category"] == "증권"].copy()
sc["성향_cot1"] = sc["cot1"].map(propensity_in)
sc["성향_cot2"] = sc["cot2"].map(propensity_in)
parsed = sc["cot2"].map(parse_allowed_grades)
sc["허용등급"] = parsed.str[0].map(lambda s: "-".join(map(str, sorted(s))) if s else "추출 실패")
sc["허용등급_출처"] = parsed.str[1]
print(sc["허용등급_출처"].value_counts())

In [ ]:
# cot1 성향 × 허용 등급 집합
pd.crosstab(sc["성향_cot1"], sc["허용등급"], margins=True)

In [ ]:
# cot2 성향 × 허용 등급 집합
pd.crosstab(sc["성향_cot2"], sc["허용등급"], margins=True)

In [ ]:
# 같은 성향(cot1 또는 cot2에서 하나만 나온 경우)인데 허용 등급 집합이 몇 종류인지
sc["성향_대표"] = np.where(sc["성향_cot1"].isin(["없음", "복수"]), sc["성향_cot2"], sc["성향_cot1"])
one = sc[~sc["성향_대표"].isin(["없음", "복수"]) & (sc["허용등급"] != "추출 실패")]
summary = one.groupby("성향_대표")["허용등급"].agg(
    건수="size", 허용등급_종류수="nunique", 최빈값=lambda s: s.value_counts().index[0],
    최빈값_비율=lambda s: round(s.value_counts(normalize=True).iloc[0] * 100, 1))
summary

In [ ]:
# 증권_03 중 cot1에 성향이 없는 건 → cot2에 성향이 있는지
s3 = sc[sc["query_group"] == "증권_03.상품 비교 추천형"]
no1 = s3[s3["성향_cot1"] == "없음"]
print("증권_03 전체:", len(s3), "/ cot1에 성향 없음:", len(no1))
no1["성향_cot2"].value_counts().to_frame("cot2 성향")

## 4. answer 속 상품명 vs `product_names`

두 가지 방법으로 본다.
- (A) answer에서 큰따옴표 안의 문자열을 뽑아 `product_names`와 비교
- (B) `product_names`의 각 상품명(정규화)이 answer 본문(정규화)에 포함되는지

In [ ]:
# (A) 따옴표 추출
ans = cot[["split", "query_group", "cot_id", "answer", "product_names"]].copy()
ans["quoted"] = ans["answer"].map(quoted_strings)
ans["list_norm"] = ans["product_names"].map(lambda L: {norm_full(x) for x in L})
ans["quoted_norm"] = ans["quoted"].map(lambda L: {norm_full(x) for x in L})
# answer 따옴표에만 있는 이름 (목록에 없음)
ans["only_answer"] = ans.apply(lambda r: r["quoted_norm"] - r["list_norm"], axis=1)
# 목록에는 있으나 answer 따옴표에 없는 이름
ans["only_list"] = ans.apply(lambda r: r["list_norm"] - r["quoted_norm"], axis=1)
print("따옴표가 하나도 없는 answer:", (ans["quoted"].map(len) == 0).sum())
print("answer 따옴표에만 있는 이름이 1개 이상인 CoT:", (ans["only_answer"].map(len) > 0).sum())
print("목록에만 있는 이름이 1개 이상인 CoT:", (ans["only_list"].map(len) > 0).sum())

In [ ]:
# (B) product_names 각 상품명이 answer 본문에 포함되는지 (둘 다 정규화)
ans["answer_norm"] = ans["answer"].map(norm_full)
ans["list_missing"] = ans.apply(lambda r: [x for x in r["product_names"] if norm_full(x) not in r["answer_norm"]], axis=1)
ans["n_missing"] = ans["list_missing"].map(len)
print("목록 상품 중 answer 본문에 없는 상품이 1개 이상인 CoT:", (ans["n_missing"] > 0).sum())
pd.crosstab(ans["query_group"], ans["n_missing"])

In [ ]:
# 샘플: (A) answer 따옴표에만 있는 이름 5건, (B) 본문에 없는 목록 상품 5건
print("=== (A) answer 따옴표에만 있는 이름 ===")
for _, r in ans[ans["only_answer"].map(len) > 0].sample(5, random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 따옴표 원문:", [q for q in r["quoted"] if norm_full(q) in r["only_answer"]][:2])
print()
print("=== (B) answer 본문에 없는 목록 상품 ===")
sub = ans[ans["n_missing"] > 0]
for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 없는 상품:", r["list_missing"][:2])

## 5. 메타 age·gender vs question 본문

In [ ]:
# 본문에서 연령대·성별 추출 후 메타와 비교
q = cot[["split", "query_group", "category", "cot_id", "age", "gender", "question"]].copy()
q["본문_연령대"] = q["question"].map(extract_age_decade)
q["연령_비교"] = [meta_age_match(a, d) for a, d in zip(q["age"], q["본문_연령대"])]
q["본문_성별"] = q["question"].map(extract_gender)
q["성별_비교"] = np.select([q["본문_성별"].isna(), q["본문_성별"] == "복수", q["본문_성별"] == q["gender"]],
                         ["본문에 없음", "본문에 복수", "일치"], default="불일치")
print(pd.crosstab(q["query_group"], q["연령_비교"], margins=True))
print()
print(pd.crosstab(q["query_group"], q["성별_비교"], margins=True))

In [ ]:
# 불일치 샘플 (연령 5건, 성별 5건): 본문에서 표현 주변 60자
import re
def around(text, pat, n=30):
    m = re.search(pat, str(text))
    return str(text)[max(0, m.start() - n): m.end() + n] if m else ""

print("=== 연령 불일치 ===")
for _, r in q[q["연령_비교"] == "불일치"].sample(5, random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 메타={r['age']} / 본문:", around(r["question"], r"[1-9]0\s*대|\d{2}\s*(?:세|살)"))
print("=== 성별 불일치 ===")
sub = q[q["성별_비교"] == "불일치"]
for _, r in sub.sample(min(5, len(sub)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 메타={r['gender']} / 본문:", around(r["question"], r"남성|남자|여성|여자"))

## 6. 70대 이상 고객 CoT 중 1~2등급 상품 추천 (증권)

- 메타 age가 `70대`, `70대 이상`, `80대 이상`인 CoT.
- 등급은 상품 JSON 기준. 보험 상품에는 등급이 없어 증권만 본다.

In [ ]:
# 고령 메타 연령 × 증권 추천 상품의 JSON 등급
OLD = ["70대", "70대 이상", "80대 이상"]
old = rec[(rec["category"] == "증권") & rec["age"].isin(OLD)].copy()
print("대상 CoT 수:", old["cot_id"].nunique(), "/ 추천 상품 수:", len(old))
print(pd.crosstab(old["age"], old["json_grade"].fillna(-1).astype(int).replace(-1, "등급 없음/불명"), margins=True))

hi = old[old["json_grade"].isin([1, 2])]
print()
print("1~2등급 상품이 1개 이상 추천된 CoT 수:", hi["cot_id"].nunique())

In [ ]:
# 샘플 10건: 메타 연령, 성향(cot1/cot2), cot2 허용 등급, 추천 상품과 등급
hs = hi.merge(sc[["cot_id", "성향_cot1", "성향_cot2", "허용등급"]], on="cot_id", how="left")
hs.drop_duplicates("cot_id").sample(min(10, hi["cot_id"].nunique()), random_state=0)[
    ["cot_id", "age", "성향_cot1", "성향_cot2", "허용등급", "rec_name", "json_grade"]]

## 7. question 중복

- 완전 중복: 앞뒤 공백 제거·연속 공백 통일 후 문장이 똑같은 경우
- 유사 중복: 글자 2~3그램 TF-IDF 코사인 유사도 0.9 이상을 연결한 묶음 (500행 블록 단위 계산)

In [ ]:
# 완전 중복
q["q_norm"] = q["question"].str.strip().str.replace(r"\s+", " ", regex=True)
grp = q.groupby("q_norm")
dup_groups = grp.filter(lambda g: len(g) > 1)
print("완전 중복 묶음 수:", dup_groups["q_norm"].nunique(), "/ 포함 행 수:", len(dup_groups))
print("Train↔Valid에 걸친 완전 중복 묶음 수:", dup_groups.groupby("q_norm")["split"].nunique().gt(1).sum())

In [ ]:
# 유사 중복 (시간이 조금 걸린다)
q["near_group"] = near_duplicate_groups(q["question"].tolist(), threshold=0.9, block=500)
sizes = q.groupby("near_group").size()
multi = sizes[sizes > 1]
print("유사 중복 묶음 수(2건 이상):", len(multi), "/ 포함 행 수:", multi.sum())
print("최대 묶음 크기:", multi.max() if len(multi) else 0, "/ 10건 이상 묶음 수:", (multi >= 10).sum())
print()
print("묶음 크기 분포:")
print(multi.value_counts().sort_index().rename("묶음 수").to_frame().T)

In [ ]:
# 묶음이 어느 split에 걸쳐 있는지: Train만 / Valid만 / Train↔Valid
splits = q[q["near_group"].isin(multi.index)].groupby("near_group")["split"].agg(lambda s: "↔".join(sorted(set(s))))
splits.value_counts().rename("묶음 수").to_frame()

In [ ]:
# 가장 큰 유사 중복 묶음의 질문 3개 앞 120자
if len(multi) == 0:
    print("유사 중복 묶음이 없다.")
else:
    big = multi.idxmax()
    for t in q.loc[q["near_group"] == big, "question"].head(3):
        print("-", t[:120])

In [ ]:
# 참고: 임계값을 낮추면 묶음이 얼마나 생기는지 (0.9 결과를 해석할 때 비교용, 판정 아님)
for th in [0.8, 0.7]:
    lab = near_duplicate_groups(q["question"].tolist(), threshold=th, block=500)
    s = pd.Series(lab).value_counts()
    s = s[s > 1]
    print(f"유사도 {th} 이상: 묶음 {len(s)}개, 포함 행 {s.sum()}, 최대 크기 {s.max() if len(s) else 0}")

# 각 질문의 '가장 비슷한 다른 질문' 유사도 분포 (블록 계산)
from sklearn.feature_extraction.text import TfidfVectorizer
X = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit_transform(q["question"])
best = []
for s0 in range(0, X.shape[0], 500):
    sim = (X[s0:s0 + 500] @ X.T).toarray()
    for i in range(sim.shape[0]):
        sim[i, s0 + i] = 0          # 자기 자신 제외
    best.append(sim.max(axis=1))
best = np.concatenate(best)
pd.Series(best).describe(percentiles=[.5, .9, .99]).round(3)

## 8. 상품별 추천 횟수

In [ ]:
# 매칭된 상품 키 기준 추천 횟수
# 애매한 경우 처리: 이름 중복으로 후보가 여러 개면 대표 이름(display_name) 단위로 센다
tbl = idx["table"]
rec["display_name"] = rec["keys"].map(lambda k: tbl.loc[k[0], "display_name"] if k else None)
cnt = rec.dropna(subset=["display_name"]).groupby(["category", "display_name"]).size().rename("추천 횟수")
cnt.sort_values(ascending=False).head(20).to_frame()

In [ ]:
# 한 번도 추천되지 않은 상품 수 (상품 JSON 10,000개 기준)
used_keys = set(k for ks in rec["keys"] for k in ks)
tbl2 = tbl.assign(추천됨=tbl.index.isin(used_keys))
out = tbl2.groupby("domain")["추천됨"].agg(전체="size", 추천됨="sum")
out["추천 안 됨"] = out["전체"] - out["추천됨"]
print(out)
print()
print("주의: 매칭 실패(판정불가) 추천명", (rec["keys"].map(len) == 0).sum(), "건은 어떤 상품에도 연결되지 않아, 일부 상품이 '추천 안 됨'으로 세어졌을 수 있다.")

In [ ]:
# 상품 분류별 추천 안 된 비율
g = tbl2.groupby(["domain", "product_group"])["추천됨"].agg(전체="size", 추천됨="sum")
g["추천 안 된 비율(%)"] = ((1 - g["추천됨"] / g["전체"]) * 100).round(1)
g

## 9. answer가 문장 부호 없이 끝나는 건

In [ ]:
# 끝 공백·닫는 따옴표/괄호를 건너뛴 마지막 글자가 . ! ? … 。 가 아닌 건
ans["ends_ok"] = ans["answer"].map(ends_with_punct)
print("문장 부호 없이 끝나는 answer:", (~ans["ends_ok"]).sum())
print(pd.crosstab(ans["query_group"], ans["ends_ok"]))

In [ ]:
# 샘플 10건: 끝 60자
for _, r in ans[~ans["ends_ok"]].sample(min(10, (~ans["ends_ok"]).sum()), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] ...{r['answer'][-60:]!r}")

# 추가 확인 (2026-11-10)

판정 없이 사실만 확인한다.

| # | 항목 |
|---|---|
| A1 | [기준 3 핵심] 추천 상품의 JSON 등급이 cot2 허용 등급 집합 안에 있는지 |
| A2 | 성향별 허용 등급 집합 전체 분포 (질문유형별) |
| A3 | '등급 N' 표기 인식 추가 후, 수정 전 '등급 표현 없음' 건 재집계 |
| A4 | 고령 1~2등급 추천 CoT의 성향 분포와 연령 고려 표현 |
| A5 | 성별 불일치 63건에서 cot1~answer가 따르는 성별 |
| A6 | answer에 없는 목록 상품의 cot3 서술 |

## A1. 추천 상품 JSON 등급 ∈ cot2 허용 등급 집합 (증권)

In [ ]:
# CoT별 허용 등급 집합을 추천 상품 행에 붙인다
sc["허용집합"] = parsed.str[0]
a1 = rec[rec["category"] == "증권"].merge(sc[["cot_id", "허용집합", "허용등급", "허용등급_출처", "성향_대표", "cot2"]], on="cot_id")

# 애매한 경우 처리: 허용 집합 추출 실패, JSON 등급 없음(매칭 실패·등급 복수)은 안/밖을 정하지 않고 따로 센다
def in_range(r):
    if r["허용집합"] is None:
        return "허용 집합 추출 실패"
    if pd.isna(r["json_grade"]):
        return "JSON 등급 없음"
    return "범위 안" if int(r["json_grade"]) in r["허용집합"] else "범위 밖"

a1["허용범위"] = a1.apply(in_range, axis=1)
print(a1["허용범위"].value_counts())
print()
print(pd.crosstab(a1["query_group"], a1["허용범위"], margins=True))
print()
# 허용 집합을 어떤 근거로 뽑았는지(parse_allowed_grades의 근거 종류)별로 나눠 본다
# '까지 해석', '이상·이하 해석', '단일 등급만'은 해석이나 부분 정보가 들어간 집합이다
pd.crosstab(a1["허용등급_출처"], a1["허용범위"], margins=True)

In [ ]:
# 범위 밖: 성향 × (JSON 등급) 교차표
out_ = a1[a1["허용범위"] == "범위 밖"]
print("범위 밖 상품이 있는 CoT 수:", out_["cot_id"].nunique())
pd.crosstab([out_["성향_대표"], out_["허용등급"]], out_["json_grade"].astype(int))

In [ ]:
# 범위 밖 샘플 10건: cot2에서 '등급'이 들어간 문장만 보여 준다
import re
def grade_sents(t, n=220):
    s = [x for x in re.split(r"(?<=[.!?])\s+", str(t)) if "등급" in x]
    return " / ".join(s)[:n]

for _, r in out_.sample(min(10, len(out_)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] 성향={r['성향_대표']} / 허용={r['허용등급']} / JSON={int(r['json_grade'])}등급 / {r['rec_name']}")
    print("   cot2:", grade_sents(r["cot2"]))

In [ ]:
# 허용 집합 추출 실패 건의 cot2 앞부분 3건 (실패 이유 확인용)
fail = a1[a1["허용범위"] == "허용 집합 추출 실패"].drop_duplicates("cot_id")
print("허용 집합 추출 실패 CoT 수:", len(fail))
for _, r in fail.head(3).iterrows():
    print(f"[cot_id={r['cot_id']}]", str(r["cot2"])[:200])

## A2. 성향별 허용 등급 집합 분포 (질문유형별)

In [ ]:
# 성향(성향_대표)마다 허용 등급 집합 × 질문유형 건수표
for p in ["위험중립형", "안정추구형", "적극투자형", "공격투자형", "안정형"]:
    sub = sc[sc["성향_대표"] == p]
    print("=" * 80)
    print(f"{p} ({len(sub)}건)")
    print(pd.crosstab(sub["허용등급"], sub["query_group"], margins=True, margins_name="합계")
            .sort_values("합계", ascending=False))

## A3. '등급 N' 표기 인식 추가 후 재집계

`grade_after_name(..., suffix=False)`는 수정 전 방식('N등급'만 인식), 기본값 `suffix=True`는 '위험등급 2', '등급 3' 같은 표기도 인식한다.
위 2번 셀의 결과(`sec`)는 이미 수정 후 방식으로 계산된 값이다.

In [ ]:
# 수정 전 방식으로 다시 계산해 수정 후 상태와 비교
old = sec.apply(lambda r: grade_after_name(r["cot3"], r["rec_name"], r["product_names"], suffix=False), axis=1)
sec["cot3_status_old"] = old.str[0]
print(pd.crosstab(sec["cot3_status_old"], sec["cot3_status"], margins=True))

In [ ]:
# 수정 전 '등급 표현 없음'이었던 건 → 수정 후 상태와, 새로 추출된 등급의 JSON 일치 여부
was_none = sec[sec["cot3_status_old"] == "등급 표현 없음"]
print("수정 전 '등급 표현 없음':", len(was_none))
print(was_none["cot3_status"].value_counts())
newly = was_none[was_none["cot3_status"] == "추출"]
cmp_ = np.select([newly["json_grade"].isna(), newly["cot3_grade"] == newly["json_grade"]],
                 ["JSON 등급 없음", "일치"], default="불일치")
print()
print("새로 추출된 건의 JSON 비교:", pd.Series(cmp_).value_counts().to_dict())

In [ ]:
# 수정 후에도 '등급 표현 없음'인 건 샘플 3건 (상품명 뒤 150자)
for _, r in sec[sec["cot3_status"] == "등급 표현 없음"].sample(3, random_state=1).iterrows():
    print(f"[cot_id={r['cot_id']}] {r['rec_name']}")
    print("   상품명 뒤:", after_name(r["cot3"], r["rec_name"], 150).replace("\n", " "))

## A4. 고령(70대·80대 이상) CoT 중 1~2등급 추천 156건 상세

In [ ]:
# 156건과 나머지 고령 CoT(1~2등급 추천 없음)의 성향 분포
old_ids = set(old_cot := rec.loc[(rec["category"] == "증권") & rec["age"].isin(OLD), "cot_id"])
hi_ids = set(hi["cot_id"])
g = sc[sc["cot_id"].isin(old_ids)].copy()
g["구분"] = np.where(g["cot_id"].isin(hi_ids), "1~2등급 추천 있음", "1~2등급 추천 없음")
print(pd.crosstab(g["성향_대표"], g["구분"], margins=True))
print()
print("cot1 기준:")
print(pd.crosstab(g["성향_cot1"], g["구분"]))

In [ ]:
# 연령 고려 표현이 있는 건수 (필드별, 단어별)
# 애매한 경우 처리: '나이'는 '나이에 맞는'처럼 연령 고려가 아닌 문맥도 잡힐 수 있다. 문맥은 구분하지 않는다.
AGE_WORDS = ["고령", "연세", "노후", "나이", "은퇴", "연령", "노년", "시니어"]
# (sc는 cot에서 복사한 것이라 cot1~answer 컬럼이 이미 있다)
rows = []
for f in ["cot1", "cot2", "cot3", "answer"]:
    for w in AGE_WORDS:
        has = g[f].astype(str).str.contains(w)
        rows.append({"필드": f, "단어": w,
                     "1~2등급 추천 있음": has[g["구분"] == "1~2등급 추천 있음"].sum(),
                     "1~2등급 추천 없음": has[g["구분"] == "1~2등급 추천 없음"].sum()})
tbl_age = pd.DataFrame(rows).pivot(index="단어", columns="필드")
tbl_age

In [ ]:
# 네 필드 중 어디든 연령 고려 표현(위 8개 단어)이 하나라도 있는 CoT 수
pat = "|".join(AGE_WORDS)
g["연령표현_어디든"] = g[["cot1", "cot2", "cot3", "answer"]].astype(str).agg(" ".join, axis=1).str.contains(pat)
g["연령표현_사용자4개"] = g[["cot1", "cot2", "cot3", "answer"]].astype(str).agg(" ".join, axis=1).str.contains("고령|연세|노후|나이")
g.groupby("구분")[["연령표현_어디든", "연령표현_사용자4개"]].agg(["sum", "size"])

## A5. 성별 불일치 63건: 각 필드가 따르는 성별

In [ ]:
# 메타(여)와 본문(남)이 다른 63건에서 cot1·cot2·cot3·answer의 성별 언급
mm = q[q["성별_비교"] == "불일치"].merge(cot[["cot_id", "cot1", "cot2", "cot3", "answer"]], on="cot_id")
def follow(text_gender, body, meta):
    if text_gender is None:
        return "언급 없음"
    if text_gender == "복수":
        return "복수"
    return "본문 따름" if text_gender == body else ("메타 따름" if text_gender == meta else "기타")

res5 = pd.DataFrame({f: [follow(extract_gender(t), b, m) for t, b, m in zip(mm[f], mm["본문_성별"], mm["gender"])]
                     for f in ["cot1", "cot2", "cot3", "answer"]})
res5.apply(pd.Series.value_counts).fillna(0).astype(int)

In [ ]:
# cot1에서 성별을 언급한 문장 예시 3건
for _, r in mm.head(3).iterrows():
    print(f"[cot_id={r['cot_id']}] 메타={r['gender']} / 본문={r['본문_성별']} / cot1:", around(r["cot1"], r"남성|남자|여성|여자"))

## A6. answer에 없는 목록 상품 110건: cot3 서술

In [ ]:
# answer에 없는 목록 상품을 한 행씩 펼치고, cot3에서 그 상품명 뒤 구간을 본다
from src.extract import _find_all
miss = ans[ans["n_missing"] > 0][["cot_id", "query_group", "list_missing", "product_names"]].explode("list_missing")
miss = miss.merge(cot[["cot_id", "cot3"]], on="cot_id")

def cot3_seg(r, n=200):
    sp = _find_all(str(r["cot3"]), r["list_missing"])
    return str(r["cot3"])[sp[0][1]: sp[0][1] + n] if sp else None

miss["cot3_seg"] = miss.apply(cot3_seg, axis=1)
print("answer에 없는 목록 상품 수:", len(miss), "/ CoT 수:", miss["cot_id"].nunique())
print("cot3에 그 상품명이 있는 경우:", miss["cot3_seg"].notna().sum())

In [ ]:
# cot3 구간에 '제외·부적합' 계열 표현이 있는지 (단어별 건수)
# 애매한 경우 처리: '다만', '반면'은 단순 비교일 수도 있어 따로 센다. 문맥은 판단하지 않는다.
EXCL = {"제외": r"제외", "적합하지 않/부적합": r"적합하지\s*않|부적합", "부합하지 않": r"부합하지\s*않",
        "맞지 않": r"맞지\s*않", "추천하지 않/드리지 않": r"추천하지\s*않|추천드리지\s*않|권하지\s*않",
        "어렵": r"어렵", "다만": r"다만", "반면": r"반면"}
seg = miss["cot3_seg"].fillna("")
excl_tbl = pd.Series({k: seg.str.contains(p, regex=True).sum() for k, p in EXCL.items()}, name="건수")
print(excl_tbl)
strict = r"제외|적합하지\s*않|부적합|부합하지\s*않|맞지\s*않|추천하지\s*않|추천드리지\s*않|권하지\s*않"
print()
print("'다만·반면·어렵'을 뺀 제외 계열 표현이 있는 상품 수:", seg.str.contains(strict, regex=True).sum())

In [ ]:
# 샘플 10건: cot3에서 상품명 뒤 200자
for _, r in miss.sample(min(10, len(miss)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] {r['list_missing']}")
    seg_txt = r["cot3_seg"] if isinstance(r["cot3_seg"], str) else "(cot3에 상품명 없음)"
    print("   cot3:", seg_txt.replace("\n", " "))

# 추가 확인 2 (2026-11-11)

판정 없이 사실만 확인한다.

| # | 항목 |
|---|---|
| B1 | 연령대별 증권 추천 상품 중 1~2등급 비율 (고령 vs 그 외, 성향별) |
| B2 | 허용 범위 밖 33건 CoT의 예외 사유 서술 유무 |
| B3 | answer에서 추천 상품마다 숫자 위험등급이 언급되는지 |
| B4 | (D12 확정용) 보험_01 추천 상품 중 변액보험·외화보험 |

## B1. 연령대별 1~2등급 추천 비율 (증권, 상품 JSON 등급 기준)

In [ ]:
# 증권 추천 상품 중 JSON 등급이 있는 건만 대상
# 애매한 경우 처리: JSON 등급이 없는 추천(매칭 실패·등급 복수)은 분모에서 뺀다
b1 = rec[(rec["category"] == "증권") & rec["json_grade"].notna()].merge(sc[["cot_id", "성향_대표"]], on="cot_id")
b1["1~2등급"] = b1["json_grade"].isin([1, 2])
b1["연령그룹"] = np.where(b1["age"].isin(OLD), "70대 이상(70대·80대 이상)", "그 외")

def rate_tbl(df, by):
    t = df.groupby(by)["1~2등급"].agg(추천상품수="size", 등급1_2="sum")
    t["비율(%)"] = (t["등급1_2"] / t["추천상품수"] * 100).round(1)
    return t

print(rate_tbl(b1, "age"))
print()
print(rate_tbl(b1, "연령그룹"))

In [ ]:
# 성향별로 고령 vs 그 외 비교
t = rate_tbl(b1, ["성향_대표", "연령그룹"])["비율(%)"].unstack()
t.join(rate_tbl(b1, ["성향_대표", "연령그룹"])["추천상품수"].unstack(), rsuffix="_n")

## B2. 허용 범위 밖 CoT의 예외 사유 서술

방법: cot2·cot3에서 예외 사유를 나타낼 수 있는 표현을 찾는다.
- 고객 요청 계열: `요청`, `명시`, `원하`, `희망`, `선호`
- 예외·전환 계열: `예외`, `불구하고`, `그러나`, `다만`, `벗어`, `초과`

애매한 경우 처리: 단어가 있다고 예외 사유를 서술한 것으로 확정할 수 없다(예: '선호'는 일반 서술에도 나온다).
그래서 **고객 요청 계열과 예외·전환 계열이 같은 문장에 함께 나오는 경우**만 '서술 있음'으로 분류하고, 각 CoT의 해당 문장을 전부 출력해 눈으로 확인할 수 있게 한다.

In [ ]:
REQ = r"요청|명시|원하|희망|선호"
EXC = r"예외|불구하고|그러나|다만|벗어|초과"

def exception_sents(text):
    sents = re.split(r"(?<=[.!?])\s+", str(text))
    return [s for s in sents if re.search(REQ, s) and re.search(EXC, s)]

b2 = out_.drop_duplicates("cot_id")[["cot_id", "query_group", "성향_대표", "허용등급"]].merge(
    cot[["cot_id", "cot2", "cot3"]], on="cot_id")
b2["예외문장_cot2"] = b2["cot2"].map(exception_sents)
b2["예외문장_cot3"] = b2["cot3"].map(exception_sents)
b2["분류"] = np.where((b2["예외문장_cot2"].map(len) + b2["예외문장_cot3"].map(len)) > 0, "서술 있음", "서술 없음")
print("범위 밖 CoT 수:", len(b2))
print(pd.crosstab(b2["성향_대표"], b2["분류"], margins=True))

In [ ]:
# '서술 있음' CoT의 해당 문장 (앞 160자)
for _, r in b2[b2["분류"] == "서술 있음"].iterrows():
    print(f"[cot_id={r['cot_id']}] {r['성향_대표']} / 허용 {r['허용등급']}")
    for s in (r["예외문장_cot2"] + r["예외문장_cot3"])[:2]:
        print("   -", s[:160])

In [ ]:
# '서술 없음' CoT: cot2에서 '등급'이 들어간 문장 (앞 160자)
for _, r in b2[b2["분류"] == "서술 없음"].iterrows():
    print(f"[cot_id={r['cot_id']}] {r['성향_대표']} / 허용 {r['허용등급']} / cot2: {grade_sents(r['cot2'], 160)}")

## B3. answer에서 추천 상품마다 숫자 위험등급이 언급되는지

방법: cot3와 같은 함수(`grade_after_name`)를 answer에 적용한다. 함께 나열된 상품은 같은 등급 표현을 공유한다.

In [ ]:
b3 = rec.merge(cot[["cot_id", "answer", "product_names"]], on="cot_id")
res = b3.apply(lambda r: grade_after_name(r["answer"], r["rec_name"], r["product_names"]), axis=1)
b3["ans_status"] = res.str[0]
b3["ans_grade"] = res.str[1]
print(pd.crosstab(b3["query_group"], b3["ans_status"], margins=True))

In [ ]:
# answer에 등급이 추출된 증권 상품: JSON 등급과 일치 여부
s3 = b3[(b3["category"] == "증권") & (b3["ans_status"] == "추출")]
cmp3 = np.select([s3["json_grade"].isna(), s3["ans_grade"] == s3["json_grade"]], ["JSON 등급 없음", "일치"], default="불일치")
print(pd.Series(cmp3).value_counts())

# CoT 단위: 추천 상품 모두에 등급이 있음 / 일부만 / 하나도 없음 (상품명 없음은 제외하고 계산)
per = b3[b3["ans_status"] != "상품명 없음"].groupby(["category", "cot_id"])["ans_status"].agg(
    lambda s: "모두 있음" if (s == "추출").all() else ("하나도 없음" if (s != "추출").all() else "일부만"))
print()
print(per.groupby(level=0).value_counts().unstack(fill_value=0))

## B4. (D12 확정용) 보험_01 추천 상품 중 변액보험·외화보험

- 변액보험: 상품 JSON `product_type == '변액'` 또는 상품명에 '변액'
- 외화보험: 상품명·상품 설명 필드에 `달러|외화|USD|미화|위안|엔화|유로`

애매한 경우 처리: 설명 필드에 '달러' 등이 나오더라도 외화 상품이 아닌 문맥(예: 비교 설명)일 수 있다. 어느 필드에서 걸렸는지 함께 표시한다.

In [ ]:
b4 = rec[rec["query_group"] == "보험_01.연령별 및 생애주기 저축성 상품 추천형"].copy()
b4 = b4[b4["keys"].map(len) > 0]   # 매칭된 추천만
b4["key"] = b4["keys"].str[0]      # 이름 중복 시 첫 후보 (애매한 경우: 후보끼리 유형이 다를 수 있음)
info = tbl.loc[b4["key"]].reset_index()
FX = r"달러|외화|USD|미화|위안|엔화|유로"
desc_cols = ["product_full_name", "mother_product_name", "rider_type", "product_features", "tax_type",
             "minimum_guarantee", "refund_condition", "payment_conditions1"]
b4["product_type"] = info["product_type"].values
b4["변액(유형)"] = b4["product_type"].eq("변액").values
b4["변액(이름)"] = b4["rec_name"].str.contains("변액").values
fx_hit = info[desc_cols].astype(str).apply(lambda col: col.str.contains(FX, regex=True))
b4["외화_걸린필드"] = fx_hit.apply(lambda r: ",".join(r.index[r]), axis=1).values
b4["외화(이름)"] = b4["rec_name"].str.contains(FX, regex=True).values
print("보험_01 추천 상품(매칭됨):", len(b4), "/ CoT:", b4["cot_id"].nunique())
print(b4["product_type"].value_counts())
print()
summary4 = pd.DataFrame({
    "추천 건수": [b4["변액(유형)"].sum(), b4["변액(이름)"].sum(), b4["외화(이름)"].sum(), (b4["외화_걸린필드"] != "").sum()],
    "고유 상품 수": [b4.loc[b4["변액(유형)"], "key"].nunique(), b4.loc[b4["변액(이름)"], "key"].nunique(),
                b4.loc[b4["외화(이름)"], "key"].nunique(), b4.loc[b4["외화_걸린필드"] != "", "key"].nunique()],
    "CoT 수": [b4.loc[b4["변액(유형)"], "cot_id"].nunique(), b4.loc[b4["변액(이름)"], "cot_id"].nunique(),
              b4.loc[b4["외화(이름)"], "cot_id"].nunique(), b4.loc[b4["외화_걸린필드"] != "", "cot_id"].nunique()],
}, index=["변액: product_type", "변액: 상품명", "외화: 상품명", "외화: 이름·설명 필드"])
summary4

In [ ]:
# 샘플 5개씩: 변액 / 외화 (고유 상품 기준)
print("=== 변액 ===")
v = b4[b4["변액(유형)"] | b4["변액(이름)"]].drop_duplicates("key")
for _, r in v.head(5).iterrows():
    print(f"- {r['rec_name']} (product_type={r['product_type']})")
print("=== 외화 ===")
f = b4[(b4["외화_걸린필드"] != "") | b4["외화(이름)"]].drop_duplicates("key")
for _, r in f.head(5).iterrows():
    print(f"- {r['rec_name']} (걸린 필드: {r['외화_걸린필드'] or '상품명'})")

# 추가 확인 3 (2026-11-11)

판정 없이 사실만 확인한다. (C4는 D19 정의를 그대로 적용한 집계)

| # | 항목 |
|---|---|
| C1 | 파서 보완('N등급, N등급, N등급' 목록) 후 범위 밖 재집계 |
| C2 | cot1과 cot2의 투자성향이 다른 건 |
| C3 | 70대 이상 안정형 중 1~2등급 추천 건의 원문 |
| C4 | D19 기준 answer 위험 설명 충족 여부 |
| C5 | 메타 age '50대 이상', '60대 이상'의 의미 |
| C6 | 범위 밖 CoT 분류용 표 |

※ 위쪽 A1·A2·B1·B2 셀도 보완된 파서로 다시 계산된 값이다.

## C1. 파서 보완 후 범위 밖 재집계

In [ ]:
# 수정 전(D13 기록): 범위 밖 83건(CoT 33), 허용 집합 추출 실패 73건(CoT 29)
print("범위 밖 추천 상품:", (a1["허용범위"] == "범위 밖").sum(), "/ CoT:", a1.loc[a1["허용범위"] == "범위 밖", "cot_id"].nunique())
print("허용 집합 추출 실패:", (a1["허용범위"] == "허용 집합 추출 실패").sum(),
      "/ CoT:", a1.loc[a1["허용범위"] == "허용 집합 추출 실패", "cot_id"].nunique())
print()
print(sc["허용등급_출처"].value_counts())

## C2. cot1과 cot2의 투자성향이 다른 건 (증권)

In [ ]:
# 둘 다 성향 명칭이 하나씩 나온 경우만 비교 (없음·복수는 비교에서 뺀다)
both = sc[sc["성향_cot1"].isin(PROPENSITY) & sc["성향_cot2"].isin(PROPENSITY)]
diff = both[both["성향_cot1"] != both["성향_cot2"]]
print("cot1·cot2 모두 성향 1개:", len(both), "/ 서로 다름:", len(diff))
print()
print(pd.crosstab(diff["성향_cot1"], diff["성향_cot2"], margins=True))
print()
print(diff["query_group"].value_counts())

In [ ]:
# 샘플 5건: cot1과 cot2에서 성향 명칭이 들어간 문장
def prop_sent(t, n=150):
    s = [x for x in re.split(r"(?<=[.!?])\s+", str(t)) if any(p in x for p in PROPENSITY)]
    return " / ".join(s)[:n]

for _, r in diff.sample(min(5, len(diff)), random_state=0).iterrows():
    print(f"[cot_id={r['cot_id']}] cot1={r['성향_cot1']} / cot2={r['성향_cot2']}")
    print("   cot1:", prop_sent(r["cot1"]))
    print("   cot2:", prop_sent(r["cot2"]))

## C3. 70대 이상 안정형 중 1~2등급 추천 건

In [ ]:
c3 = b1[(b1["성향_대표"] == "안정형") & (b1["연령그룹"] != "그 외") & b1["1~2등급"]]
print("해당 추천 상품 수:", len(c3), "/ CoT 수:", c3["cot_id"].nunique())
for _, r in c3.merge(sc[["cot_id", "성향_cot1", "성향_cot2", "cot2"]], on="cot_id").iterrows():
    gset, kind, raw = parse_allowed_grades(r["cot2"], with_text=True)
    print("=" * 90)
    print(f"[cot_id={r['cot_id']}] {r['age']} / cot1={r['성향_cot1']} / cot2={r['성향_cot2']}")
    print(f"   추천: {r['rec_name']} → JSON {int(r['json_grade'])}등급")
    print(f"   허용 집합: {sorted(gset) if gset else None} ({kind}) / 원문: {raw}")
    print("   cot2 등급 문장:", grade_sents(r["cot2"], 300))

## C4. D19 기준 answer 위험 설명 충족 여부 (증권)

D19: 추천 상품마다 숫자 등급 또는 공식 명칭(매우높은위험~매우낮은위험) 중 하나가 answer에 있으면 충족.

애매한 경우 처리: answer에 상품명이 없는 추천(상품명 없음)은 그 상품에 대한 설명을 찾을 수 없으므로 **미충족**으로 센다. 몇 건인지 따로 표시한다.

In [ ]:
from src.extract import risk_mention_after_name
c4 = b3[b3["category"] == "증권"].copy()
c4["mention"] = c4.apply(lambda r: risk_mention_after_name(r["answer"], r["rec_name"], r["product_names"]), axis=1)
c4["충족"] = c4["mention"].isin(["숫자 등급", "공식 명칭", "둘 다"])
print(c4["mention"].value_counts())
print()
print("추천 상품 기준 충족:", c4["충족"].sum(), "/", len(c4), f"({c4['충족'].mean()*100:.1f}%)")

In [ ]:
# CoT 단위: 전부 충족 / 일부 / 전무
cot_lvl = c4.groupby("cot_id")["충족"].agg(lambda s: "전부 충족" if s.all() else ("전무" if not s.any() else "일부"))
print(cot_lvl.value_counts())
print()
# 비교: B3(숫자 등급만) 기준
print(pd.crosstab(c4.drop_duplicates("cot_id").set_index("cot_id").loc[cot_lvl.index, "query_group"], cot_lvl, margins=True))
print()
print("'상품명 없음' 추천이 있어 미충족이 된 CoT 수:", c4.loc[c4["mention"] == "상품명 없음", "cot_id"].nunique())

## C5. 메타 age '50대 이상', '60대 이상'

B1 표의 `50대 이상`(4건), `60대 이상`(3건)은 **추천 상품 수**다. 이 값을 가진 증권 CoT 수는 아래와 같다.
메타 age에는 `50대`와 `50대 이상`, `60대`와 `60대 이상`이 따로 있다(02 노트북 4단계). 두 값이 어떻게 다른지는 문서에 설명이 없어 **불확실**하다.

In [ ]:
print(sc["age"].value_counts().sort_index())
print()
for a in ["50대 이상", "60대 이상"]:
    sub = sc[sc["age"] == a]
    for _, r in sub.iterrows():
        print(f"[{a}] cot_id={r['cot_id']} / 본문 연령 표현:", around(r["question"], r"[1-9]0\s*대|\d{2}\s*(?:세|살)", 25))

## C6. 범위 밖 CoT 분류용 표

- 한 행 = 범위 밖 추천이 있는 CoT 1건. 범위 밖 상품이 여러 개면 이어 붙인다.
- `data/review/범위밖_분류표.csv`로 저장한다 (원문 문장이 들어 있어 git 제외 폴더인 `data/` 아래에 둔다).
- `내 분류` 칸은 비워 둔다.

In [ ]:
out2 = a1[a1["허용범위"] == "범위 밖"]
rows = []
for cid, g in out2.groupby("cot_id"):
    r0 = sc[sc["cot_id"] == cid].iloc[0]
    gset, kind, raw = parse_allowed_grades(r0["cot2"], with_text=True)
    rows.append({
        "cot_id": cid,
        "질문유형": r0["query_group"],
        "성향_cot1": r0["성향_cot1"],
        "성향_cot2": r0["성향_cot2"],
        "허용등급(파싱)": "-".join(map(str, sorted(gset))) if gset else "",
        "허용등급 원문": " | ".join(raw),
        "범위 밖 추천 상품": " | ".join(g["rec_name"]),
        "추천 등급(JSON)": " | ".join(str(int(x)) for x in g["json_grade"]),
        "cot2 등급 문장": grade_sents(r0["cot2"], 1000),
        "내 분류": "",
    })
review = pd.DataFrame(rows)
out_dir = ROOT / "data" / "review"
out_dir.mkdir(parents=True, exist_ok=True)
review.to_csv(out_dir / "범위밖_분류표.csv", index=False, encoding="utf-8-sig")   # 엑셀에서 한글이 깨지지 않게 utf-8-sig
print("저장:", out_dir / "범위밖_분류표.csv", "/ 행 수:", len(review))
review.drop(columns=["cot2 등급 문장", "범위 밖 추천 상품"])

## 02b 정리 (직접 작성)

1. 2번 등급 불일치는 어떤 패턴인가? 03 판정(기준 3)에서 상품 등급의 기준을 무엇으로 할 것인가? (cot3 표기 / JSON)
2. 3번에서 같은 성향인데 허용 등급이 다른 경우를 어떻게 볼 것인가?
3. 4·5·9번 불일치·미완결 건을 정제 데이터셋에서 어떻게 다룰 것인가?
4. 6번 고령자 고위험 추천을 품질 기준에 넣을 것인가? 근거 규정은 무엇인가? (docs/sources.md)
5. 7번 중복을 Train/Valid 구성과 정제 데이터셋에서 어떻게 처리할 것인가?
6. decisions.md에 추가할 항목:
